# Dynamics Transformer — perturbation × time → per-gene response

Physics-informed gene-transformer on **LINCS L1000**. **GPU runtime required.** The LINCS table is cached to Drive, so after the first build a fresh runtime **skips the 20 GB download**.

On a big GPU (RTX 6000 / A100): raise `DTF_BATCH` (512–1024), `DTF_EPOCHS`, and `LINCS_MAX_SIGS` for more coverage.

In [ ]:
# 1) GPU + clone + deps + mount Drive
import torch, subprocess, os, sys
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -> Runtime > Change runtime type > GPU')
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull'])
subprocess.run('pip install -q cmapPy h5py pandas openpyxl', shell=True)
from google.colab import drive; drive.mount('/content/drive')
DRV='/content/drive/MyDrive/virtual_cell_data/dynamics_transformer'; os.makedirs(DRV, exist_ok=True)
print('cwd', os.getcwd(), '| Drive cache:', DRV)

In [ ]:
# 2) PHYSICS features (half-lives) + EGF test course  (small; always re-fetch)
import subprocess, sys
subprocess.run([sys.executable,'colab/fetch_dynamics_data.py'])
subprocess.run([sys.executable,'colab/fetch_timecourse.py'])

In [ ]:
# 3) LINCS table: LOAD FROM DRIVE if cached, else build (20 GB) + cache to Drive
import subprocess, sys, os, shutil
os.makedirs('outputs/orphan', exist_ok=True)
cache=f'{DRV}/lincs_train.npz'; local='outputs/orphan/lincs_train.npz'
if os.path.exists(cache):
    shutil.copy(cache, local); print('loaded lincs_train.npz from Drive (%d MB) -> skipped 20 GB download' % (os.path.getsize(local)//1048576))
else:
    os.environ['LINCS_GSE']='GSE92742'; os.environ['LINCS_MAX_SIGS']='120000'
    subprocess.run([sys.executable,'colab/fetch_lincs.py'])
    if os.path.exists(local): shutil.copy(local, cache); print('built + cached to Drive for next time')

In [ ]:
# 4) TRAIN (streams each epoch; bf16 autocast). Big GPU -> raise batch/epochs.
import subprocess, sys, os
os.environ['DTF_EPOCHS']='15'; os.environ['DTF_DIM']='128'; os.environ['DTF_LAYERS']='3'
os.environ['DTF_BATCH']='512'      # RTX6000/A100 can go 512-1024; drop to 256/96 if OOM
os.environ['DTF_SPLIT']='signature'  # seen perts, new time/cell (our goal). 'perturbation' = cold-start
subprocess.run([sys.executable,'colab/dynamics_transformer.py'])

In [ ]:
# 5) STAGE-1 TEST: query at EGF's minute-timepoints -> does it BEAT 0.25?
import subprocess, sys
subprocess.run([sys.executable,'colab/eval_transformer_egf.py'])

In [ ]:
# 6) save checkpoint + evals (+ table) to Drive
import shutil, os, json
for f in ['dynamics_transformer.pt','dynamics_transformer_eval.json','eval_transformer_egf.json','lincs_train.npz']:
    p=f'outputs/orphan/{f}'
    if os.path.exists(p): shutil.copy(p, DRV); print('saved', f, os.path.getsize(p)//1048576, 'MB')